# Experiment 03 — Encoding Strategies (BCE Standard)

**Goal:** Compare Angle, Amplitude, and Data Re-uploading using Binary Cross-Entropy loss and proper probability calibration.

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.metrics import roc_auc_score
from utils.data_utils import load_higgs

N_SAMPLES = 1000
N_FEATURES = 2

def train_vqc(circuit_fn, n_qubits, scale_range, label):
    X_train, X_val, X_test, y_train, y_val, y_test = load_higgs(
        path='../data/HIGGS.csv.gz', n_samples=N_SAMPLES, n_features=N_FEATURES, scale_range=scale_range
    )
    
    def vqc_prob(w, x, b):
        return (circuit_fn(w, x) + b + 1.0) / 2.0
    
    def bce_loss(weights, bias, X, y):
        probs = pnp.array([vqc_prob(weights, x, bias) for x in X])
        probs = pnp.clip(probs, 1e-15, 1.0 - 1e-15)
        return -pnp.mean(y * pnp.log(probs) + (1 - y) * pnp.log(1 - probs))
    
    pnp.random.seed(42)
    w = pnp.array(pnp.random.uniform(0, 2*np.pi, (2, n_qubits, 3)), requires_grad=True)
    b = pnp.array(0.0, requires_grad=True)
    opt = qml.AdamOptimizer(stepsize=0.05)
    
    for epoch in range(20):
        idx = np.random.permutation(len(X_train))
        for start in range(0, len(X_train), 32):
            Xb, yb = X_train[idx][start:start+32], y_train[idx][start:start+32].astype(float)
            w, b = opt.step(lambda w_, b_: bce_loss(w_, b_, Xb, yb), w, b)
            
    test_probs = np.array([float(vqc_prob(w, x, b)) for x in X_test])
    auc = roc_auc_score(y_test, test_probs)
    print(f"{label:12s} | Test AUC: {auc:.4f}")